# Governance

Route by where the data is allowed to go, not only by how hard the task is. Thresholds stay in code.


In [1]:
import sys
from datetime import date
from pathlib import Path
import json
import re
import statistics

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from langchain_typesafe import Choice, Noul, NoulCriteria, Score
from jev_examples.settings import ask, ask_many, draft, jev_model, openai_ready, show, typesafe_ready
from jev_examples.sample_data import (
    corpus_docs,
    customers,
    emails,
    load_json,
    lookup_order,
    open_incidents,
    order,
    products,
    read_text,
    ticket,
    tickets,
)

print("Jev model:", jev_model())
print("Jev key set:", typesafe_ready())
print("OpenAI key set:", openai_ready())


Jev model: jev-latest
Jev key set: True
OpenAI key set: True


## 31. Which endpoint is allowed to see this prompt?

A public policy question can use a dev endpoint. A prompt that names an unreleased project or a customer contract stays on an approved one. This cell names the endpoint. It does not call it.


In [2]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    questions = {
        "contains_internal": Noul(instructions="Does `prompt` include internal project names, customer contracts, or credentials?"),
        "complexity": Score(
            instructions="How much reasoning does `prompt` need?",
            criteria=["Lookup", "Moderate", "Deep"],
        ),
    }
    for item in load_json("work.json")["prompts"]:
        response = ask(item, questions)
        show(response)
        internal = response.nouls["contains_internal"].noul > 0.35
        heavy = response.scores["complexity"].score > 1.2
        if internal and heavy:
            route = "approved_big"
        elif internal:
            route = "approved_fast"
        else:
            route = "dev_free"
        print(item["id"], "->", route)


model: jev-1.13.0
  noul   contains_internal: 0.02
  score  complexity: 0.02  ~ Lookup
public -> dev_free


model: jev-1.13.0
  noul   contains_internal: 0.97
  score  complexity: 0.33  ~ Lookup
internal -> approved_fast


**What you should see.** The refund-window question should be `dev_free`. The Harbor / Acme prompt should be an approved endpoint.


## 32. Classify a document before it enters context

A tagline is public. A row that includes a shopper email and a key is not.


In [3]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    questions = {
        "sensitivity": Score(
            instructions="What is the sensitivity of `document`?",
            criteria=["Public", "Internal", "Confidential", "Restricted"],
        ),
        "has_credentials": Noul(instructions="Does `document` contain a password, API key, or token?"),
    }
    for doc in load_json("work.json")["documents"]:
        response = ask({"document": doc["text"]}, questions)
        show(response)
        if response.nouls["has_credentials"].noul > 0.5 or response.scores["sensitivity"].score > 2.4:
            route = "withhold"
        elif response.scores["sensitivity"].score > 1.4:
            route = "taint_the_run"
        else:
            route = "allow"
        print(doc["name"], "->", route)


model: jev-1.13.0
  noul   has_credentials: 0.01
  score  sensitivity: 0.00  ~ Public
tagline -> allow


model: jev-1.13.0
  noul   has_credentials: 0.95
  score  sensitivity: 2.04  ~ Confidential
customer-export -> withhold


**What you should see.** The tagline should be allowed. The customer export with a key should be withheld.


## 33. Which business unit pays for this run?

Low confidence becomes `unattributed` instead of a guess.


In [4]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    units = load_json("work.json")["business_units"]
    questions = {"unit": Choice(instructions="Which business unit does this request serve?", criteria=units)}
    for ticket_id in ("T-340", "T-360", "T-370"):
        response = ask(ticket(ticket_id)["body"], questions)
        show(response)
        answer = response.choices["unit"]
        label = answer.choice if answer.confidence > 0.55 else "unattributed"
        print(ticket_id, "->", label)


model: jev-1.13.0
  choice unit: wholesale  (confidence 0.42)
T-340 -> unattributed


model: jev-1.13.0
  choice unit: infra  (confidence 1.00)
T-360 -> infra


model: jev-1.13.0
  choice unit: retail_support  (confidence 0.57)
T-370 -> retail_support


**What you should see.** The Acme invoice should be wholesale. The checkout 500s note should be infra. The tent-size idea is the soft one.


## 34. Grade a trace on atomic questions

Each Noul is a column. This replaces one vague 'was the agent good?' score.


In [5]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    questions = {
        "unneeded_tool_call": Noul(instructions="Did `trace` call a tool the final answer did not need?"),
        "hedged_without_cause": Noul(instructions="Does `final_answer` refuse or hedge even though `trace` had the fact?"),
        "invented_fact": Noul(instructions="Does `final_answer` assert something not in `trace`?"),
        "answer_quality": Score(
            instructions="How well does `final_answer` satisfy `spec`?",
            criteria=["Fails", "Partial", "Meets"],
        ),
    }
    for item in load_json("agent.json")["traces"]:
        response = ask(item, questions)
        show(response)
        print("quality:", round(response.scores["answer_quality"].score, 2))


model: jev-1.13.0
  noul   unneeded_tool_call: 0.11
  noul   hedged_without_cause: 0.03
  noul   invented_fact: 0.29
  score  answer_quality: 1.49  ~ Partial
quality: 1.49


model: jev-1.13.0
  noul   unneeded_tool_call: 0.93
  noul   hedged_without_cause: 0.30
  noul   invented_fact: 0.94
  score  answer_quality: 0.10  ~ Fails
quality: 0.1


**What you should see.** The trace that looked up A-118 and said it shipped should meet the spec. The trace that searched products and then hedged should not.


## 35. Did a new prompt version change behavior?

Run the same questions on two small sets of outputs. Alert when the average moves.


In [6]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    outputs = load_json("agent.json")["prompt_outputs"]
    questions = {
        "asks_question": Noul(instructions="Does `output` ask the user a question instead of answering?"),
        "apologizes": Noul(instructions="Does `output` contain an apology?"),
        "uses_bullets": Noul(instructions="Is `output` formatted mainly as bullet points?"),
    }
    left = ask_many([{"state": {"output": text}, "questions": questions} for text in outputs["v1"]])
    right = ask_many([{"state": {"output": text}, "questions": questions} for text in outputs["v2"]])
    for name in questions:
        before = statistics.mean(item.nouls[name].noul for item in left)
        after = statistics.mean(item.nouls[name].noul for item in right)
        print(name, "delta", round(after - before, 2), "regressed" if abs(after - before) > 0.2 else "stable")


asks_question delta 0.48 regressed
apologizes delta 0.98 regressed
uses_bullets delta 0.4 regressed


**What you should see.** Version 2 apologizes, asks a question, and uses bullets. Those three deltas should move. Version 1 is two short answers.


## 36. Which permission scopes does this turn need?

One Noul per scope. Mint a token with only the ones that clear the bar.


In [7]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    scopes = {
        "read_orders": "Read an order",
        "refund": "Send money back",
        "send_email": "Email another person",
        "manage_catalog": "Edit products",
    }
    for task in load_json("work.json")["tasks"]:
        questions = {
            "needs_%s" % name: Noul(instructions="To complete `task`, would the assistant need to: %s?" % text)
            for name, text in scopes.items()
        }
        response = ask({"task": task}, questions)
        show(response)
        needed = [name for name in scopes if response.nouls["needs_%s" % name].noul > 0.6]
        print(task, "->", needed)


model: jev-1.13.0
  noul   needs_read_orders: 0.95
  noul   needs_refund: 0.04
  noul   needs_send_email: 0.10
  noul   needs_manage_catalog: 0.03
Read the status of order A-118 and tell the shopper. -> ['read_orders']


model: jev-1.13.0
  noul   needs_read_orders: 0.74
  noul   needs_refund: 0.48
  noul   needs_send_email: 0.31
  noul   needs_manage_catalog: 0.06
Cancel order A-290 and email the shopper that it was cancelled. -> ['read_orders']


**What you should see.** Reading A-118 should need `read_orders` only. Cancelling A-290 and emailing the shopper should need more than a read.
